In [ ]:
pip install requests beautifulsoup4 pandas lxml


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import json
from datetime import date, timedelta
import re

def get_page_html(url):
  headers = {'User-Agent':('Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
            'AppleWebKit/537.36 (KHTML, like Gecko) '
            'Chrome/91.0.4472.124 Safari/537.36')}

  try:
    response = requests.get(url, headers = headers, timeout = 10)
    if response.status_code == 200:
      return response.text
    else:
      print(f"Failed to fetch {url} - Status: {response.status_code}")
      return None
  except Exception as e:
    print(f"Error fetching {url}: {e}")
    return None


def get_article_links(listing_page_url):
    html = get_page_html(listing_page_url)
    if html is None:
        return []
    soup = BeautifulSoup(html, 'lxml')
    article_links = []

    article_url_pattern = re.compile(r'/\w+/\d+$')

    seen = set()
    for a_tag in soup.find_all('a', href=True):
        href = a_tag['href']

        if not href.startswith('http'):
            href = 'https://www.vanglaini.org' + href

        path = href.replace('https://www.vanglaini.org', '')
        if article_url_pattern.search(path) and href not in seen:
            seen.add(href)
            article_links.append(href)
    return article_links


def scrape_article(url):
  html = get_page_html(url)
  if html is None:
    return None

  soup = BeautifulSoup(html, 'lxml')

  title = None
  title_tag = soup.find('title')
  if title_tag:
    title = title_tag.get_text(strip = True)

  body = None
  body_div = soup.find('div')
  if body_div:
    paragraph = body_div.find_all('p')
    body = ' '.join([p.get_text(strip = True) for p in paragraph])
  date = None
  time_tag = soup.find('time')
  if time_tag:
    date = time_tag.get('datetime') or time_tag.get_text(strip = True)
  if not date:
    meta_date = soup.find('meta', {'property' : 'article:published_time'})
    if meta_date:
      date = meta_date.get('content')

  if title and body and len(body)>50:
    return {
        'url' : url,
        'title' : title,
        'body' : body,
        'full_text' : title + ' ' + body,
        'date' : date,
        'source' : url.split('/')[2],
        'label' : None
    }
  return None


def scrape_news_site(base_listing_url, num_pages = 5, label_hint = None):
  all_articles = []
  for page_num in range(1, num_pages + 1):
    target_date = date.today() - timedelta(days=page_num - 1)
    listing_url = f"{base_listing_url}?date={target_date.strftime('%Y-%m-%d')}"

    print(f"  Fetching edition for date: {target_date}")
    links = get_article_links(listing_url)
    print(f"  Found {len(links)} article links")

    for link in links:
       article_data = scrape_article(link)
       if article_data:
          article_data['label'] = label_hint
          all_articles.append(article_data)
          time.sleep(1.5)

    time.sleep(2)

  return all_articles


def save_data(articles, filename = 'mizo_news_raw.csv'):
  if not articles:
    print("No articles to save")
    return

  df = pd.DataFrame(articles)
  df = df.drop_duplicates(subset = ['url'])
  df.to_csv(filename, index = False, encoding = 'utf-8-sig')
  print(f"\nSaved {len(df)} articles to {filename}")
  print(f"Columns: {list(df.columns)}")
  print(df.head())


if __name__ == "__main__":
  all_data = []

  print("--scraping reliable source--")
  real_articles = scrape_news_site(
      base_listing_url='https://www.vanglaini.org/',
      num_pages=60,
      label_hint = 'REAL'
  )
  all_data.extend(real_articles)
  print(f"Collected {len(real_articles)} articles from reliable source")

  save_data(all_data, 'mizo_news_raw.csv')


In [ ]:
import pandas as pd

df = pd.read_csv('mizo_news_raw.csv', encoding = 'utf-8-sig')

print(f"Total articles collected : {len(df)}")
print(f"Columns : {list(df.columns)}" )

print(f"\n Missing values per column: ")
print(df.isnull().sum())

for i, row in df.head(3).iterrows() :
  print(f"\n{'=' * 50}")
  print(f"URL  : {row['url']}")
  print(f"Title: {row['title'][:80]}")
  print(f"Body : {str(row['body'])[:200]}")
  print(f"Date : {row['date']}")


In [ ]:
import pandas as pd
import re
import unicodedata
import html

def remove_html_tags(text) :
  return re.sub(r'<.*?>', '', str(text))

def decode_html_entities(text) :
  return html.unescape(text)

def normalize_unicode(text) :
  return unicodedata.normalize('NFC', text)

def remove_urls(text) :
  return re.sub(r'https?://\S+|www\.\S+', '', text)

def remove_extra_whitespace(text) :
  return re.sub(r'\s+', ' ', text).strip()

def clean_title(title):
    if not isinstance(title, str):
        return None
    for sep in [' | ', ' - ', ' – ', ' — ']:
        if sep in title:
            title = max(title.split(sep), key=len).strip()
            break
    return title

def is_valid_article(title, body):
    if not isinstance(title, str) or len(title.strip()) < 5:
        return False
    if not isinstance(body, str) or len(body.strip()) < 100:
        return False
    if len(body.split()) < 20:
        return False
    return True

def preprocess_text(text):
    if not isinstance(text, str):
        return None
    text = remove_html_tags(text)
    text = decode_html_entities(text)
    text = normalize_unicode(text)
    text = remove_urls(text)
    text = remove_extra_whitespace(text)
    if len(text.strip()) < 10:
        return None
    return text

def combine_text(row):
    title = str(row['title_clean']) if pd.notna(row['title_clean']) else ''
    body  = str(row['body_clean'])  if pd.notna(row['body_clean'])  else ''
    combined = (title + ' ' + body).strip()
    return combined if combined else None

def preprocess_dataset(input_path, output_path):
    df = pd.read_csv(input_path, encoding='utf-8-sig')
    print(f"Loaded {len(df)} articles")

    df['title_clean']     = df['title'].apply(clean_title).apply(preprocess_text)
    df['body_clean']      = df['body'].apply(preprocess_text)
    df['full_text_clean'] = df.apply(combine_text, axis=1)
    df['word_count']      = df['body_clean'].apply(
                                lambda x: len(str(x).split()) if pd.notna(x) else 0)

    before = len(df)
    df['is_valid'] = df.apply(
        lambda row: is_valid_article(row['title_clean'], row['body_clean']), axis=1)
    df = df[df['is_valid']].drop(columns=['is_valid']).copy()
    df = df.drop_duplicates(subset=['full_text_clean'])
    print(f"Kept {len(df)} articles (removed {before - len(df)} short/duplicate/empty)")

    df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"Saved to '{output_path}'")
    print(df['word_count'].describe().round(1))
    return df

def prepare_annotation_file(input_path, output_path, sample_size=None):
    df = pd.read_csv(input_path, encoding='utf-8-sig')
    if sample_size and sample_size < len(df):
        df = df.sample(n=sample_size, random_state=42)
    out = df[['url', 'title_clean', 'body_clean']].copy()
    out['label']      = ''
    out['confidence'] = ''
    out['reason']     = ''
    out = out.reset_index(drop=True)
    out.to_csv(output_path, index=True, encoding='utf-8-sig')
    print(f"Annotation file ready: {len(out)} articles → '{output_path}'")

if __name__ == "__main__":
    df_clean = preprocess_dataset('mizo_news_raw.csv', 'mizo_news_clean.csv')
    prepare_annotation_file('mizo_news_clean.csv', 'mizo_news_to_annotate.csv', sample_size=300)

    print("\n=== SAMPLE ===")
    for _, row in df_clean.head(3).iterrows():
        print(f"\nTitle : {row['title_clean']}")
        print(f"Body  : {str(row['body_clean'])[:200]}...")
        print(f"Words : {row['word_count']}")


In [ ]:
import pandas as pd
import re
import random
import copy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

MIZO_POLITICIANS = [
    'Zoramthanga', 'Lalthanhawla', 'Lalduhoma', 'Vanlalthlana',
    'Lalchamliana', 'Zodintluanga', 'Pu Lalduhoma', 'Pu Zoramthanga'
]

MIZO_LOCATIONS = [
    'Aizawl', 'Lunglei', 'Champhai', 'Serchhip', 'Kolasib',
    'Lawngtlai', 'Mamit', 'Saitual', 'Khawzawl', 'Hnahthial'
]

MIZO_ORGANIZATIONS = [
    'ZPM', 'MNF', 'Congress', 'BJP',
    'Young Mizo Association', 'Mizo Hmeichhe Insuihkhawm Pawl',
    'Mizoram Upa Pawl', 'Synod', 'MHIP', 'YMA'
]

NUMBER_PATTERNS = [
    r'vaibelchhia\s*\d+',
    r'vbc\s*\d+',
    r'nuai\s*\d+',
    r'\d+\s*za zela',
    r'\d+\s*%',
    r'\d{4}',
    r'\d+\s*km',
    r'\d+'
]

def mutate_numbers(text):
    def replace_number(match):
        original = match.group()
        num_match = re.search(r'\d+', original)
        if not num_match:
            return original
        num = int(num_match.group())
        factor = random.choice([2, 3, 4, 5])
        new_num = num * factor if random.random() > 0.5 else max(1, num // factor)
        return original.replace(num_match.group(), str(new_num))
    modified = text
    for pattern in NUMBER_PATTERNS:
        modified = re.sub(pattern, replace_number, modified)
    return modified

def swap_entity(text, entity_list):
    found = None
    for entity in entity_list:
        if entity in text:
            found = entity
            break
    if not found:
        return text
    replacements = [e for e in entity_list if e != found]
    return text.replace(found, random.choice(replacements), 1)

def insert_negation(text):
    pairs = [
        ('a ṭha', 'a ṭha lo'),
        ('a lo ni', 'a lo ni lo'),
        ('a puang', 'a puang lo'),
        ('chhuah', 'chhuah loh'),
        ('puang', 'puang lo'),
        ('hawng', 'hawng lo'),
        ('a dik', 'a dik lo'),
        ('a phal', 'a phal lo'),
        ('a nuam', 'a nuam lo'),
        ('tling', 'tling lo'),
        ('a siam', 'a siam lo'),
        ('a ti', 'a ti lo'),
        ('fimkhur', 'fimkhur lo'),
        ('sawi', 'sawi lo'),
        ('thiam', 'thiam loh'),
        ('avangin', 'loh avangin'),
        ('passed', 'not passed'),
        ('approved', 'not approved'),
        ('complete', 'incomplete'),
        ('success', 'failure'),
        ('won', 'lost'),
    ]
    for positive, negative in pairs:
        if positive in text:
            return text.replace(positive, negative, 1)
    return text

def reverse_outcome(text):
    pairs = [
        ('pung', 'tlem'), ('tlem', 'pung'),
        ('chak', 'tlawm'), ('tlawm', 'chak'),
        ('thuchhuak', 'thuchhuak lo'),
        ('pawm', 'hnawl'), ('hnawl', 'pawm'),
        ('hawng', 'khar'), ('hawnna', 'kharna'),
        ('khar', 'hawng'),
        ('a ṭhat', 'a ṭhat loh'),
        ('a dik', 'a dik lo'),
        ('WhatsApp', 'Telegram'),
        ('Mizoram', 'Nagaland'),
        ('India', 'China'),
        ('US', 'Canada'), ('Europe', 'Africa')
    ]
    for a, b in pairs:
        if a in text:
            return text.replace(a, b, 1)
    return text

def swap_location(text):
    for loc in MIZO_LOCATIONS:
        if loc in text:
            replacements = [l for l in MIZO_LOCATIONS if l != loc]
            return text.replace(loc, random.choice(replacements), 1)
    return text

def swap_organization(text):
    for org in MIZO_ORGANIZATIONS:
        if org in text:
            replacements = [o for o in MIZO_ORGANIZATIONS if o != org]
            return text.replace(org, random.choice(replacements), 1)
    return text

def shuffle_sentences(text):
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if len(s.strip()) > 15]
    if len(sentences) < 3:
        return text
    random.shuffle(sentences)
    return ' '.join(sentences)

def inject_foreign_sentence(text, all_real_bodies):
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if len(s.strip()) > 15]
    if len(sentences) < 2:
        return text
    foreign_body = random.choice(all_real_bodies)
    foreign_sents = [s.strip() for s in re.split(r'(?<=[.!?])\s+', str(foreign_body)) if len(s.strip()) > 15]
    if not foreign_sents:
        return text
    insert_pos = random.randint(1, len(sentences) - 1)
    sentences.insert(insert_pos, random.choice(foreign_sents))
    return ' '.join(sentences)

def safe_text(value):
    if value is None:
        return ''
    s = str(value).strip()
    if s.lower() == 'nan' or s == '':
        return ''
    return s

def generate_synthetic_fake(real_article_dict, perturbation_type, all_real_bodies):
    fake = copy.deepcopy(real_article_dict)

    title = safe_text(fake.get('title_clean', ''))
    body  = safe_text(fake.get('body_clean', ''))

    if len(body) < 20:
        return None

    original_body = body

    if perturbation_type == 'number':
        title = mutate_numbers(title)
        body  = mutate_numbers(body)
    elif perturbation_type == 'entity':
        title = swap_entity(title, MIZO_POLITICIANS)
        body  = swap_entity(body, MIZO_POLITICIANS)
    elif perturbation_type == 'negation':
        title = insert_negation(title)
        body  = insert_negation(body)
    elif perturbation_type == 'outcome':
        title = reverse_outcome(title)
        body  = reverse_outcome(body)
    elif perturbation_type == 'location':
        title = swap_location(title)
        body  = swap_location(body)
    elif perturbation_type == 'organization':
        title = swap_organization(title)
        body  = swap_organization(body)
    elif perturbation_type == 'shuffle':
        body  = shuffle_sentences(body)
    elif perturbation_type == 'inject':
        body  = inject_foreign_sentence(body, all_real_bodies)

    if body == original_body:
        body = mutate_numbers(body)
    if body == original_body:
        body = shuffle_sentences(body)
    if body == original_body:
        return None

    fake['title_clean']     = title
    fake['body_clean']      = body
    fake['full_text_clean'] = (title + ' ' + body).strip()
    fake['label']           = 'FAKE'
    fake['source']          = 'synthetic'
    fake['perturbation']    = perturbation_type
    fake['original_url']    = safe_text(fake.get('url', ''))
    fake['url']             = 'synthetic_' + safe_text(fake.get('url', ''))

    if len(fake['full_text_clean']) < 20:
        return None

    return fake

def generate_synthetic_dataset(real_csv_path, output_path, num_fakes=None):
    df_real = pd.read_csv(real_csv_path, encoding='utf-8-sig')
    df_real = df_real.dropna(subset=['body_clean'])
    df_real = df_real[df_real['body_clean'].astype(str).str.len() > 50]
    print(f"Loaded {len(df_real)} valid real articles")

    if num_fakes is None:
        num_fakes = len(df_real)

    all_real_bodies = df_real['body_clean'].astype(str).tolist()

    replace = num_fakes > len(df_real)
    sample  = df_real.sample(n=num_fakes, replace=replace, random_state=42)

    perturbation_types = ['number', 'entity', 'negation', 'outcome', 'location', 'organization', 'shuffle', 'inject']
    fake_articles = []
    skipped = 0

    for i, (_, row) in enumerate(sample.iterrows()):
        ptype = perturbation_types[i % len(perturbation_types)]
        fake  = generate_synthetic_fake(row.to_dict(), ptype, all_real_bodies)
        if fake is not None:
            fake_articles.append(fake)
        else:
            skipped += 1

    print(f"Generated {len(fake_articles)} fake articles ({skipped} skipped)")

    df_fake = pd.DataFrame(fake_articles)

    df_real = df_real.copy()
    df_real['perturbation']    = 'none'
    df_real['original_url']    = df_real['url']
    df_real['full_text_clean'] = (
        df_real['title_clean'].apply(safe_text) + ' ' +
        df_real['body_clean'].apply(safe_text)
    ).str.strip()

    df_combined = pd.concat([df_real, df_fake], ignore_index=True)
    df_combined = df_combined.dropna(subset=['full_text_clean'])
    df_combined = df_combined[df_combined['full_text_clean'].str.len() > 20]
    df_combined = df_combined.sample(frac=1, random_state=42).reset_index(drop=True)

    df_combined.to_csv(output_path, index=False, encoding='utf-8-sig')

    print(f"\nDataset saved to '{output_path}'")
    print(f"Label distribution:\n{df_combined['label'].value_counts()}")
    print(f"\nPerturbation breakdown:\n{df_combined['perturbation'].value_counts()}")

    real_indexed = df_combined[df_combined['label'] == 'REAL'].set_index('url')
    fake_indexed = df_combined[df_combined['label'] == 'FAKE'].copy()
    fake_indexed['lookup_url'] = fake_indexed['original_url']
    fake_indexed = fake_indexed.set_index('lookup_url')

    common = real_indexed.index.intersection(fake_indexed.index)
    print(f"\nMatched pairs for similarity check: {len(common)}")

    if len(common) > 10:
        n = min(len(common), 100)
        sample_urls = random.sample(list(common), n)
        real_texts = real_indexed.loc[sample_urls, 'full_text_clean'].astype(str).tolist()
        fake_texts = fake_indexed.loc[sample_urls, 'full_text_clean'].astype(str).tolist()

        vec = TfidfVectorizer().fit(real_texts + fake_texts)
        sims = [
            cosine_similarity(
                vec.transform([real_texts[i]]),
                vec.transform([fake_texts[i]])
            )[0][0]
            for i in range(n)
        ]

        print(f"\nSimilarity check (TRUE matched pairs):")
        print(f"  Mean : {np.mean(sims):.4f}  (target: 0.50+)")
        print(f"  Min  : {np.min(sims):.4f}")
        print(f"  Max  : {np.max(sims):.4f}")
    else:
        print("Not enough matched pairs found for similarity check.")

    return df_combined

if __name__ == "__main__":
    df = generate_synthetic_dataset(
        real_csv_path='mizo_news_clean.csv',
        output_path='mizo_news_with_synthetic_fake.csv'
    )
    print("\n=== SAMPLE COMPARISON ===")
    real_sample = df[df['label'] == 'REAL'].iloc[0]
    fake_sample  = df[df['label'] == 'FAKE'].iloc[0]
    print(f"\nREAL title : {real_sample['title_clean']}")
    print(f"REAL body  : {str(real_sample['body_clean'])[:200]}")
    print(f"\nFAKE title : {fake_sample['title_clean']}")
    print(f"FAKE body  : {str(fake_sample['body_clean'])[:200]}")
    print(f"Perturbation: {fake_sample['perturbation']}")


In [ ]:
import pandas as pd
import re

def strip_boilerplate_completely(text):
    if not isinstance(text, str):
        return text
    text = re.sub(r'Download the app by scanning the QR code', '', text)
    text = re.sub(r'Email:\s*contact@vanglaini\.org', '', text)
    text = re.sub(r'For technical support\s*-?\s*vanglaini@arsi\.in', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_real_raw = pd.read_csv('mizo_news_clean.csv', encoding='utf-8-sig')
print(f"STEP 1 — Loaded {len(df_real_raw)} rows")

before = df_real_raw['body_clean'].astype(str).str.count('Download the app').sum()

df_real_raw['body_clean'] = df_real_raw['body_clean'].apply(strip_boilerplate_completely)
df_real_raw['title_clean'] = df_real_raw['title_clean'].apply(strip_boilerplate_completely)
df_real_raw['full_text_clean'] = (
    df_real_raw['title_clean'].astype(str) + ' ' + df_real_raw['body_clean'].astype(str)
).str.strip()

df_real_raw = df_real_raw[df_real_raw['body_clean'].astype(str).str.len() > 50]

after = df_real_raw['body_clean'].astype(str).str.count('Download the app').sum()
print(f"Boilerplate before: {before}, after: {after}")

df_real_raw.to_csv('mizo_news_clean.csv', index=False, encoding='utf-8-sig')
print("Saved cleaned mizo_news_clean.csv successfully!\n")


In [ ]:
import re
import random
import copy
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

random.seed(42)
np.random.seed(42)

MIZO_POLITICIANS = ['Zoramthanga', 'Lalthanhawla', 'Lalduhoma', 'Vanlalthlana',
                    'Lalchamliana', 'Zodintluanga', 'Pu Lalduhoma', 'Pu Zoramthanga']
MIZO_LOCATIONS = ['Aizawl', 'Lunglei', 'Champhai', 'Serchhip', 'Kolasib',
                  'Lawngtlai', 'Mamit', 'Saitual', 'Khawzawl', 'Hnahthial']
MIZO_ORGANIZATIONS = ['ZPM', 'MNF', 'Congress', 'BJP', 'Young Mizo Association',
                      'Mizo Hmeichhe Insuihkhawm Pawl', 'Mizoram Upa Pawl', 'Synod', 'MHIP', 'YMA']

def swap_entity(text, entity_list):
    found = None
    for entity in entity_list:
        if entity in text:
            found = entity
            break
    if not found:
        return text
    replacements = [e for e in entity_list if e != found]
    return text.replace(found, random.choice(replacements), 1)

def insert_negation(text):
    pairs = [
        ('a ṭha', 'a ṭha lo'), ('a lo ni', 'a lo ni lo'), ('a puang', 'a puang lo'),
        ('chhuah', 'chhuah loh'), ('puang', 'puang lo'), ('hawng', 'hawng lo'),
        ('a dik', 'a dik lo'), ('a phal', 'a phal lo'), ('a nuam', 'a nuam lo'),
        ('tling', 'tling lo'), ('a siam', 'a siam lo'), ('a ti', 'a ti lo'),
        ('fimkhur', 'fimkhur lo'), ('sawi', 'sawi lo'), ('thiam', 'thiam loh'),
        ('avangin', 'loh avangin'), ('passed', 'not passed'), ('approved', 'not approved'),
        ('complete', 'incomplete'), ('success', 'failure'), ('won', 'lost'),
        ('remti', 'remti lo'),
        ('tluang', 'tluang lo'),
        ('thleng', 'thleng lo'),
        ('hriatpuina', 'hriatpuina tel lo'), ('kal', 'kal lo'), ('hawnna', 'kharna'), ('kharna', 'hawnna'), ]
    for positive, negative in pairs:
        if positive in text:
            return text.replace(positive, negative, 1)
    return text

def reverse_outcome(text):
    pairs = [
        ('pung', 'tlem'), ('tlem', 'pung'), ('chak', 'tlawm'), ('tlawm', 'chak'),
        ('thuchhuak', 'thuchhuak lo'), ('pawm', 'hnawl'), ('hnawl', 'pawm'),
        ('hawng', 'khar'), ('hawnna', 'kharna'), ('khar', 'hawng'),
        ('a ṭhat', 'a ṭhat loh'), ('a dik', 'a dik lo'), ('WhatsApp', 'Telegram'),
        ('Mizoram', 'Nagaland'), ('India', 'China'), ('US', 'Canada'), ('Europe', 'Africa'), ('Chief Minister', 'Finance Minister'), ('Prime Minister', 'President'), ('President', 'Prime MInister'),
        ('Vanglaini', 'Zozam Times'), ('thum', 'li'), ('li', 'nga'), ('nuai', 'sing'), ]
    for a, b in pairs:
        if a in text:
            return text.replace(a, b, 1)
    return text

def swap_location(text):
    for loc in MIZO_LOCATIONS:
        if loc in text:
            replacements = [l for l in MIZO_LOCATIONS if l != loc]
            return text.replace(loc, random.choice(replacements), 1)
    return text

def swap_organization(text):
    for org in MIZO_ORGANIZATIONS:
        if org in text:
            replacements = [o for o in MIZO_ORGANIZATIONS if o != org]
            return text.replace(org, random.choice(replacements), 1)
    return text

def shuffle_sentences(text):
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if len(s.strip()) > 15]
    if len(sentences) < 3:
        return text
    random.shuffle(sentences)
    return ' '.join(sentences)

def inject_foreign_sentence(text, all_real_bodies):
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if len(s.strip()) > 15]
    if len(sentences) < 2:
        return text
    foreign_body = random.choice(all_real_bodies)
    foreign_sents = [s.strip() for s in re.split(r'(?<=[.!?])\s+', str(foreign_body)) if len(s.strip()) > 15]
    if not foreign_sents:
        return text
    insert_pos = random.randint(1, len(sentences) - 1)
    sentences.insert(insert_pos, random.choice(foreign_sents))
    return ' '.join(sentences)

def safe_text(value):
    if value is None:
        return ''
    s = str(value).strip()
    if s.lower() == 'nan' or s == '':
        return ''
    return s

def generate_synthetic_fake(real_article_dict, perturbation_type, all_real_bodies):
    fake = copy.deepcopy(real_article_dict)
    title = safe_text(fake.get('title_clean', ''))
    body  = safe_text(fake.get('body_clean', ''))
    if len(body) < 20:
        return None
    original_body = body

    if perturbation_type == 'entity':
        title = swap_entity(title, MIZO_POLITICIANS)
        body  = swap_entity(body, MIZO_POLITICIANS)
    elif perturbation_type == 'negation':
        title = insert_negation(title)
        body  = insert_negation(body)
    elif perturbation_type == 'outcome':
        title = reverse_outcome(title)
        body  = reverse_outcome(body)
    elif perturbation_type == 'location':
        title = swap_location(title)
        body  = swap_location(body)
    elif perturbation_type == 'organization':
        title = swap_organization(title)
        body  = swap_organization(body)
    elif perturbation_type == 'shuffle':
        body  = shuffle_sentences(body)
    elif perturbation_type == 'inject':
        body  = inject_foreign_sentence(body, all_real_bodies)

    if body == original_body:
        body = shuffle_sentences(body)
    if body == original_body:
        return None

    fake['title_clean']     = title
    fake['body_clean']      = body
    fake['full_text_clean'] = (title + ' ' + body).strip()
    fake['label']           = 'FAKE'
    fake['source']          = 'synthetic'
    fake['perturbation']    = perturbation_type
    fake['original_url']    = safe_text(fake.get('url', ''))
    fake['url']             = 'synthetic_' + safe_text(fake.get('url', ''))

    if len(fake['full_text_clean']) < 20:
        return None
    return fake

def generate_synthetic_dataset(real_csv_path, output_path, num_fakes=None):
    df_real = pd.read_csv(real_csv_path, encoding='utf-8-sig')
    df_real = df_real.dropna(subset=['body_clean'])
    df_real = df_real[df_real['body_clean'].astype(str).str.len() > 50]
    print(f"STEP 2 — Loaded {len(df_real)} valid real articles")

    if num_fakes is None:
        num_fakes = len(df_real)
    all_real_bodies = df_real['body_clean'].astype(str).tolist()
    replace = num_fakes > len(df_real)
    sample  = df_real.sample(n=num_fakes, replace=replace, random_state=42)

    perturbation_types = ['entity', 'negation', 'outcome', 'location', 'organization', 'shuffle', 'inject']
    fake_articles = []
    skipped = 0
    for i, (_, row) in enumerate(sample.iterrows()):
        ptype = perturbation_types[i % len(perturbation_types)]
        fake  = generate_synthetic_fake(row.to_dict(), ptype, all_real_bodies)
        if fake is not None:
            fake_articles.append(fake)
        else:
            skipped += 1
    print(f"Generated {len(fake_articles)} fake articles ({skipped} skipped)")

    df_fake = pd.DataFrame(fake_articles)
    df_real = df_real.copy()
    df_real['perturbation']    = 'none'
    df_real['original_url']    = df_real['url']
    df_real['full_text_clean'] = (
        df_real['title_clean'].apply(safe_text) + ' ' + df_real['body_clean'].apply(safe_text)
    ).str.strip()

    df_combined = pd.concat([df_real, df_fake], ignore_index=True)
    df_combined = df_combined.dropna(subset=['full_text_clean'])
    df_combined = df_combined[df_combined['full_text_clean'].str.len() > 20]
    df_combined = df_combined.sample(frac=1, random_state=42).reset_index(drop=True)
    df_combined.to_csv(output_path, index=False, encoding='utf-8-sig')

    print(f"Dataset saved to '{output_path}'")
    print(f"Label distribution:\n{df_combined['label'].value_counts()}")
    print(f"Perturbation breakdown:\n{df_combined['perturbation'].value_counts()}")
    print(f"Boilerplate check (should be 0): {df_combined['full_text_clean'].astype(str).str.count('Download the app').sum()}")

    real_indexed = df_combined[df_combined['label'] == 'REAL'].set_index('url')
    fake_indexed = df_combined[df_combined['label'] == 'FAKE'].copy()
    fake_indexed['lookup_url'] = fake_indexed['original_url']
    fake_indexed = fake_indexed.set_index('lookup_url')
    common = real_indexed.index.intersection(fake_indexed.index)
    if len(common) > 10:
        n = min(len(common), 100)
        sample_urls = random.sample(list(common), n)
        real_texts = real_indexed.loc[sample_urls, 'full_text_clean'].astype(str).tolist()
        fake_texts = fake_indexed.loc[sample_urls, 'full_text_clean'].astype(str).tolist()
        vec = TfidfVectorizer().fit(real_texts + fake_texts)
        sims = [cosine_similarity(vec.transform([real_texts[i]]), vec.transform([fake_texts[i]]))[0][0] for i in range(n)]
        print(f"Similarity check (TRUE matched pairs): mean={np.mean(sims):.4f}\n")
    return df_combined

generate_synthetic_dataset('mizo_news_clean.csv', 'mizo_news_with_synthetic_fake.csv')


In [ ]:
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit, cross_val_score
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.pipeline import Pipeline

def mask_numbers(text):
    text = re.sub(r'\d{4}', ' YEAR ', text)
    text = re.sub(r'\d+', ' NUM ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df = pd.read_csv('mizo_news_with_synthetic_fake.csv', encoding='utf-8-sig')

df['label'] = df['label'].astype(str).str.strip().str.upper()
df = df[df['label'].isin(['REAL', 'FAKE'])].reset_index(drop=True)

print("--- Sanity Check on Raw Labels ---")
print(df['label'].value_counts())
print("---------------------------------\n")

df['label_num'] = df['label'].map({'FAKE': 0, 'REAL': 1})
df['full_text_clean'] = df['full_text_clean'].dropna().astype(str).apply(mask_numbers)

df['group_id'] = df['original_url'].astype(str)

X = df['full_text_clean']
y = df['label_num'].astype(int)
groups = df['group_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train = groups.iloc[train_idx]

print(f"STEP 3 — Group-Safe Dataset Split:")
print(f"Training set size : {len(X_train)} (Unique parent stories: {groups_train.nunique()})")
print(f"Test set size     : {len(X_test)}\n")

def make_tfidf():
    return TfidfVectorizer(
        analyzer='word',
        ngram_range=(1, 3),
        max_features=25000,
        min_df=2,
        sublinear_tf=True
    )

pipelines = {
    'Logistic Regression': Pipeline([('tfidf', make_tfidf()), ('clf', LogisticRegression(max_iter=1000, random_state=42))]),
    'Linear SVM':           Pipeline([('tfidf', make_tfidf()), ('clf', LinearSVC(max_iter=2000, random_state=42))]),
    'Random Forest':        Pipeline([('tfidf', make_tfidf()), ('clf', RandomForestClassifier(n_estimators=100, random_state=42))]),
}

results = {}
for model_name, pipeline in pipelines.items():
    print(f"{'='*55}\n  {model_name}\n{'='*55}")

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, average='weighted')

    print(f"Accuracy : {acc:.3f}  ({acc*100:.1f}%)")
    print(f"F1 Score : {f1:.3f}")
    print(classification_report(y_test, y_pred, target_names=['FAKE', 'REAL']))

    from sklearn.model_selection import LeaveOneGroupOut, GroupKFold
    gkf = GroupKFold(n_splits=5)
    cv_scores = cross_val_score(pipeline, X_train, y_train, groups=groups_train, cv=gkf, scoring='f1_weighted')
    print(f"Group-CV F1 mean: {cv_scores.mean():.3f}, std: {cv_scores.std():.3f}\n")

    results[model_name] = {'accuracy': acc, 'f1': f1, 'cv_mean': cv_scores.mean()}

print(f"{'='*55}\n  FINAL GENUINE PERFORMANCE SUMMARY\n{'='*55}")
for name, r in results.items():
    print(f"{name:<22} acc={r['accuracy']:.3f}  f1={r['f1']:.3f}  Group-CV={r['cv_mean']:.3f}")

lr_pipeline   = pipelines['Logistic Regression']
feature_names = lr_pipeline.named_steps['tfidf'].get_feature_names_out()
coefficients  = lr_pipeline.named_steps['clf'].coef_[0]
sorted_idx    = np.argsort(coefficients)

print("\nTop 15 Predictors → FAKE (Negative Weights):")
for idx in sorted_idx[:15]:
    print(f"  {repr(feature_names[idx]):<25} weight: {coefficients[idx]:.3f}")

print("\nTop 15 Predictors → REAL (Positive Weights):")
for idx in sorted_idx[-15:]:
    print(f"  {repr(feature_names[idx]):<25} weight: {coefficients[idx]:.3f}")


In [ ]:
import os
import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score, f1_score,
    classification_report, confusion_matrix
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

CONFIG = {
    'MODELS': {
        'XLM-RoBERTa': 'xlm-roberta-base',
        'mBERT':        'bert-base-multilingual-cased',
        'MizBERT':      'robzchhangte/MizBERT',
    },
    'DATA_PATH':     'mizo_news_with_synthetic_fake.csv',
    'OUTPUT_DIR':    'phase4_outputs',
    'MAX_LEN':       256,
    'BATCH_SIZE':    8,
    'EPOCHS':        5,
    'LEARNING_RATE': 2e-5,
    'TEST_SIZE':     0.2,
    'VAL_SIZE':      0.1,
    'RANDOM_STATE':  42,
}

os.makedirs(CONFIG['OUTPUT_DIR'], exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Running on: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

def mask_numbers(text):
    text = re.sub(r'\d{4}', ' YEAR ', str(text))
    text = re.sub(r'\d+', ' NUM ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("\nLoading data...")
df = pd.read_csv(CONFIG['DATA_PATH'], encoding='utf-8-sig')
df['label'] = df['label'].astype(str).str.strip().str.upper()
df = df[df['label'].isin(['REAL', 'FAKE'])]
df = df.dropna(subset=['full_text_clean']).reset_index(drop=True)
df['label_num'] = df['label'].map({'FAKE': 0, 'REAL': 1})
df['full_text_clean'] = df['full_text_clean'].astype(str).apply(mask_numbers)
df['group_id'] = df['original_url'].astype(str)

print(f"Total examples: {len(df)}")
print(f"Label distribution:\n{df['label'].value_counts()}")

X      = df['full_text_clean'].values
y      = df['label_num'].values
groups = df['group_id'].values

gss_test = GroupShuffleSplit(
    n_splits=1, test_size=CONFIG['TEST_SIZE'],
    random_state=CONFIG['RANDOM_STATE'])
dev_idx, test_idx = next(gss_test.split(X, y, groups))

X_dev,  y_dev,  groups_dev  = X[dev_idx],  y[dev_idx],  groups[dev_idx]
X_test, y_test              = X[test_idx], y[test_idx]

val_fraction = CONFIG['VAL_SIZE'] / (1 - CONFIG['TEST_SIZE'])
gss_val = GroupShuffleSplit(
    n_splits=1, test_size=val_fraction,
    random_state=CONFIG['RANDOM_STATE'])
train_idx, val_idx = next(gss_val.split(X_dev, y_dev, groups_dev))

X_train, y_train = X_dev[train_idx], y_dev[train_idx]
X_val,   y_val   = X_dev[val_idx],   y_dev[val_idx]

print(f"\nTrain : {len(X_train)}")
print(f"Val   : {len(X_val)}")
print(f"Test  : {len(X_test)}")

class MizoNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }

def train_one_epoch(model, dataloader, optimizer, scheduler, device):
    model.train()
    total_loss    = 0
    correct_preds = 0
    total_preds   = 0

    for batch in dataloader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss   = outputs.loss
        logits = outputs.logits

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss    += loss.item()
        predictions    = torch.argmax(logits, dim=1)
        correct_preds += (predictions == labels).sum().item()
        total_preds   += labels.size(0)

    return total_loss / len(dataloader), correct_preds / total_preds

def evaluate(model, dataloader, device):
    model.eval()
    total_loss = 0
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            total_loss += outputs.loss.item()
            predictions = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(predictions.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(dataloader)
    acc      = accuracy_score(all_labels, all_preds)
    f1       = f1_score(all_labels, all_preds, average='weighted')

    return avg_loss, acc, f1, all_preds, all_labels

def run_model(model_name, model_id, config, device):
    print(f"\n{'═'*60}")
    print(f"  MODEL : {model_name}")
    print(f"  ID    : {model_id}")
    print(f"{'═'*60}")

    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    print("Loading model...")
    model = AutoModelForSequenceClassification.from_pretrained(
        model_id,
        num_labels=2,
        ignore_mismatched_sizes=True
    )
    model = model.to(device)

    total_params = sum(p.numel() for p in model.parameters())
    print(f"Parameters: {total_params:,}")

    train_dataset = MizoNewsDataset(X_train, y_train, tokenizer, config['MAX_LEN'])
    val_dataset   = MizoNewsDataset(X_val,   y_val,   tokenizer, config['MAX_LEN'])
    test_dataset  = MizoNewsDataset(X_test,  y_test,  tokenizer, config['MAX_LEN'])

    train_loader = DataLoader(train_dataset, batch_size=config['BATCH_SIZE'], shuffle=True)
    val_loader   = DataLoader(val_dataset,   batch_size=config['BATCH_SIZE'], shuffle=False)
    test_loader  = DataLoader(test_dataset,  batch_size=config['BATCH_SIZE'], shuffle=False)

    optimizer = AdamW(
        model.parameters(),
        lr=config['LEARNING_RATE'],
        weight_decay=0.01
    )

    total_steps  = len(train_loader) * config['EPOCHS']
    warmup_steps = int(0.1 * total_steps)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss':   [], 'val_acc':   [], 'val_f1': []
    }

    best_val_f1    = 0
    best_model_path = os.path.join(
        config['OUTPUT_DIR'], f"{model_name}_best.pt")

    for epoch in range(1, config['EPOCHS'] + 1):
        print(f"\n  Epoch {epoch}/{config['EPOCHS']}")

        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, scheduler, device)

        val_loss, val_acc, val_f1, _, _ = evaluate(
            model, val_loader, device)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)

        print(f"    Train → loss: {train_loss:.4f}  acc: {train_acc:.3f}")
        print(f"    Val   → loss: {val_loss:.4f}  acc: {val_acc:.3f}  F1: {val_f1:.3f}")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), best_model_path)
            print(f"    ✓ Best model saved (val F1 = {val_f1:.3f})")

    print(f"\n  Loading best checkpoint...")
    model.load_state_dict(
        torch.load(best_model_path, map_location=device))

    test_loss, test_acc, test_f1, test_preds, test_labels = evaluate(
        model, test_loader, device)

    print(f"\n  ── FINAL TEST RESULTS: {model_name} ──")
    print(f"  Accuracy : {test_acc:.4f}  ({test_acc*100:.2f}%)")
    print(f"  F1 Score : {test_f1:.4f}")
    print(f"\n  Detailed Report:")
    print(classification_report(
        test_labels, test_preds, target_names=['FAKE', 'REAL']))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    epochs_range = range(1, config['EPOCHS'] + 1)

    axes[0].plot(epochs_range, history['train_loss'], label='Train', marker='o')
    axes[0].plot(epochs_range, history['val_loss'],   label='Val',   marker='o')
    axes[0].set_title(f'{model_name} — Loss per Epoch')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs_range, history['train_acc'], label='Train', marker='o')
    axes[1].plot(epochs_range, history['val_acc'],   label='Val',   marker='o')
    axes[1].set_title(f'{model_name} — Accuracy per Epoch')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    curve_path = os.path.join(
        config['OUTPUT_DIR'], f"{model_name}_learning_curves.png")
    plt.savefig(curve_path, dpi=150)
    plt.show()

    return {
        'model_name':  model_name,
        'test_acc':    test_acc,
        'test_f1':     test_f1,
        'test_preds':  test_preds,
        'test_labels': test_labels,
        'history':     history,
        'best_val_f1': best_val_f1,
    }

def plot_confusion_matrices(results_list, output_dir):
    n    = len(results_list)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 5))
    if n == 1:
        axes = [axes]

    for ax, result in zip(axes, results_list):
        cm = confusion_matrix(result['test_labels'], result['test_preds'])
        ax.imshow(cm, cmap='Blues')
        ax.set_title(
            f"{result['model_name']}\nF1={result['test_f1']:.3f}", fontsize=12)
        ax.set_xlabel('Predicted')
        ax.set_ylabel('Actual')
        ax.set_xticks([0, 1])
        ax.set_yticks([0, 1])
        ax.set_xticklabels(['FAKE', 'REAL'])
        ax.set_yticklabels(['FAKE', 'REAL'])
        for i in range(2):
            for j in range(2):
                ax.text(j, i, str(cm[i, j]),
                        ha='center', va='center', fontsize=16,
                        fontweight='bold',
                        color='white' if cm[i,j] > cm.max()/2 else 'black')

    plt.suptitle(
        'Confusion Matrices — Phase 4 Transformer Models', fontsize=13)
    plt.tight_layout()
    path = os.path.join(output_dir, 'confusion_matrices_phase4.png')
    plt.savefig(path, dpi=150)
    plt.show()

if __name__ == "__main__":
    all_results = []

    for model_name, model_id in CONFIG['MODELS'].items():
        result = run_model(model_name, model_id, CONFIG, device)
        all_results.append(result)

    print("\n" + "═"*60)
    print("  PHASE 4 — FINAL MODEL COMPARISON")
    print("═"*60)
    print(f"{'Model':<20} {'Test Acc':>10} {'Test F1':>10} {'Best Val F1':>12}")
    print("-"*60)
    for r in all_results:
        print(f"{r['model_name']:<20} {r['test_acc']:>10.4f} "
              f"{r['test_f1']:>10.4f} {r['best_val_f1']:>12.4f}")

    print(f"\nPhase 3 baseline (Linear SVM): Test F1 = 0.586, Group-CV F1 = 0.591")
    best = max(all_results, key=lambda x: x['test_f1'])
    print(f"Best transformer: {best['model_name']} (Test F1 = {best['test_f1']:.4f})")
    print(f"Improvement over baseline: {best['test_f1'] - 0.591:+.4f}")

    plot_confusion_matrices(all_results, CONFIG['OUTPUT_DIR'])

    summary = pd.DataFrame([{
        'model':       r['model_name'],
        'test_acc':    round(r['test_acc'],    4),
        'test_f1':     round(r['test_f1'],     4),
        'best_val_f1': round(r['best_val_f1'], 4),
    } for r in all_results])

    path = os.path.join(CONFIG['OUTPUT_DIR'], 'model_comparison_phase4.csv')
    summary.to_csv(path, index=False)
    print(f"\nSaved → {path}")
